In [21]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import json
import re
from pathlib import Path
from tqdm.notebook import tqdm

BASE_URL   = "https://www.micuro.it"
LIST_URL   = "https://www.micuro.it/strutture/regione/lombardia"
LIST_PARAMS = {"g": "1", "l": "Lombardia", "s": ""}

DELAY_MIN  = 0.8
DELAY_MAX  = 1.5

OUTPUT_DIR = Path()
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "it-IT,it;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

session = requests.Session()
session.headers.update(HEADERS)

In [22]:
def fetch(url: str, params: dict = None) -> BeautifulSoup | None:
    try:
        resp = session.get(url, params=params, timeout=15)
        resp.raise_for_status()
        html = resp.content.decode("utf-8", errors="replace")
        return BeautifulSoup(html, "lxml")
    except requests.RequestException as e:
        print(f" Errore su {url}: {e}")
    return None


def polite_sleep():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

In [23]:
def parse_list_page(soup: BeautifulSoup) -> list[dict]:
    strutture = []
    for h2 in soup.select("h2"):
        a_tag = h2.find(
            "a",
            href=lambda h: h and h.startswith("/strutture/") and "/regione/" not in h
        )
        if not a_tag:
            continue

        nome = a_tag.get_text(strip=True)
        href = BASE_URL + a_tag["href"]

        indirizzo = ""
        for sib in h2.next_siblings:
            text = sib.get_text(" ", strip=True) if hasattr(sib, "get_text") else str(sib).strip()
            if not text:
                continue
            if not indirizzo and re.search(r'\d{5}', text):
                indirizzo = re.sub(r'\bMappa\b.*', '', text).strip()
                break

        strutture.append({"nome": nome, "indirizzo": indirizzo, "href": href})

    return strutture


def get_total_pages(soup: BeautifulSoup) -> int:
    max_page = 1
    for a in soup.select("a[href*='page=']"):
        m = re.search(r'page=(\d+)', a["href"])
        if m:
            max_page = max(max_page, int(m.group(1)))
    return max_page

In [24]:
soup_p1 = fetch(LIST_URL, params=LIST_PARAMS)
assert soup_p1 is not None, "Impossibile scaricare la prima pagina!"

total_pages = get_total_pages(soup_p1)
print(f"Pagine totali: {total_pages}")

all_strutture = parse_list_page(soup_p1)
polite_sleep()

for page in tqdm(range(2, total_pages + 1), desc="Pagine lista"):
    soup = fetch(LIST_URL, params={**LIST_PARAMS, "page": str(page)})
    if soup is None:
        print(f"  SKIP pagina {page}")
    else:
        all_strutture.extend(parse_list_page(soup))
    polite_sleep()

seen = set()
strutture_uniq = []
for s in all_strutture:
    if s["href"] not in seen:
        seen.add(s["href"])
        strutture_uniq.append(s)

print(f"Strutture uniche: {len(strutture_uniq)}")

Pagine totali: 7


Pagine lista:   0%|          | 0/6 [00:00<?, ?it/s]

Strutture uniche: 134


In [25]:
DIMENSIONI_GENERALI = {
    "da consigliare a parenti e amici",
    "pulizia",
    "rispetto della privacy",
    "qualità dell'ospitalità e della struttura",
    "disponibilità e gentilezza del personale medico",
    "disponibilità e gentilezza del personale non medico",
    "chiarezza delle informazioni mediche ricevute",
    "chiarezza delle informazioni amministrative e organizzative",
    "qualità dei pasti",
    "gestione delle visite dei parenti",
}
SKIP_LABELS = {"valutazione globale", "inserisci valutazione"}

def parse_valutazioni(soup: BeautifulSoup) -> dict:
    out = {
        "rating_globale": None,
        "n_valutazioni": None,
        "rating_dimensioni": {},
        "rating_per_area": {},
    }

    section = soup.find(id="health-facilities-reviews")
    if section is None:
        for heading in soup.find_all(["h2", "h3", "h4", "h5"]):
            if "valutazion" in heading.get_text().lower() and "utenti" in heading.get_text().lower():
                section = heading.find_parent("section") or heading.find_parent("div")
                break
    if section is None:
        return out

    lines = [l.strip() for l in section.get_text("\n").split("\n") if l.strip()]

    for i, line in enumerate(lines):
        # Caso A: "5.0 /5" o "5.0/5" sulla stessa riga
        m = re.match(r'^(\d+[.,]\d+)\s*/\s*5$', line)
        if not m:
            # Caso B: "5.0" su una riga e "/5" sulla successiva (due <span>)
            m1 = re.match(r'^(\d+[.,]\d+)$', line)
            if m1 and i + 1 < len(lines) and re.match(r'^/\s*5$', lines[i + 1]):
                m = m1
        if m:
            out["rating_globale"] = float(m.group(1).replace(",", "."))
            for j in range(i + 1, min(i + 5, len(lines))):
                m2 = re.match(r'^(\d+)\s+valutazion', lines[j], re.IGNORECASE)
                if m2:
                    out["n_valutazioni"] = int(m2.group(1))
                    break
            break

    i = 0
    while i < len(lines):
        line = lines[i]
        label, voto = None, None

        if i + 1 < len(lines):
            m = re.match(r'^(\d+[.,]\d+)$', lines[i + 1])
            if m:
                val = float(m.group(1).replace(",", "."))
                if 0.0 <= val <= 5.0:
                    label, voto = line, val
                    i += 2

        if label is None:
            m = re.match(r'^(.+?)\s+(\d+[.,]\d+)$', line)
            if m:
                val = float(m.group(2).replace(",", "."))
                if 0.0 <= val <= 5.0:
                    label, voto = m.group(1).strip(), val
            i += 1

        if label is None or voto is None:
            continue

        label_lower = label.lower()
        if label_lower in SKIP_LABELS:
            continue
        elif label_lower in DIMENSIONI_GENERALI:
            out["rating_dimensioni"][label] = voto
        else:
            out["rating_per_area"][label] = voto

    return out

def parse_detail_page(soup: BeautifulSoup, href: str) -> dict:
    
    result = {"href": href}

    aree = []
    for a in soup.find_all("a", href=True):
        if not re.search(r'/area-specialistica/[\w-]+$', a["href"]):
            continue
        area_nome = next((s.strip() for s in a.strings if s.strip()), "")
        if area_nome and area_nome not in aree:
            aree.append(area_nome)
    result["aree_specialistiche"] = aree

    result.update(parse_valutazioni(soup))
    return result

In [26]:
CHECKPOINT_DETAILS = OUTPUT_DIR / "strutture_dettagli.json"

details_map = {}

todo = [s for s in strutture_uniq]
print(f"Strutture da scaricare: {len(todo)}")

for i, struttura in enumerate(tqdm(todo, desc="Dettagli strutture")):
    href = struttura["href"]
    soup = fetch(href)
    if soup is None:
        print(f"  SKIP: {href}")
        details_map[href] = {"href": href, "error": "fetch_failed"}
    else:
        details_map[href] = parse_detail_page(soup, href)
    polite_sleep()

Strutture da scaricare: 134


Dettagli strutture:   0%|          | 0/134 [00:00<?, ?it/s]

In [27]:
rows = []
for struttura in strutture_uniq:
    href = struttura["href"]
    d = details_map.get(href, {})
    rows.append({
        "nome"                  : struttura["nome"],
        "indirizzo"             : struttura["indirizzo"],
        "href"                  : href,
        "rating_globale"        : d.get("rating_globale"),
        "n_valutazioni"         : d.get("n_valutazioni"),
        "n_aree"                : len(d.get("aree_specialistiche", [])),
        "aree_specialistiche"   : ", ".join(d.get("aree_specialistiche", [])),
        "rating_dimensioni_json": json.dumps(d.get("rating_dimensioni", {}), ensure_ascii=False),
        "rating_per_area_json"  : json.dumps(d.get("rating_per_area",   {}), ensure_ascii=False),
    })

df = pd.DataFrame(rows)
print(f"DataFrame: {len(df)} righe, {len(df.columns)} colonne")
print(f"Con rating_globale valorizzato: {df['rating_globale'].notna().sum()}")
print(f"Con rating per area:            {(df['rating_per_area_json'] != '{}').sum()}")
df.head()

DataFrame: 134 righe, 9 colonne
Con rating_globale valorizzato: 101
Con rating per area:            89


,nome,indirizzo,href,rating_globale,n_valutazioni,n_aree,aree_specialistiche,rating_dimensioni_json,rating_per_area_json
0,Ospedale Carlo Ondoli di Angera - ASST Sette L...,"Via Bordini, 9 - 21021\n Angera (VA)",https://www.micuro.it/strutture/ospedale-carlo...,5.0,3.0,14,"Analisi di Laboratorio, Cardiologia, Chirurgia...","{""Da consigliare a parenti e amici"": 5.0, ""Pul...","{""Ortopedia e Traumatologia"": 5.0}"
1,Ospedale dei Bambini di Brescia - ASST Spedali...,"Via del Medolo, 2 - 25123\n Brescia (BS)",https://www.micuro.it/strutture/ospedale-dei-b...,5.0,2.0,15,"Analisi di Laboratorio, Cardiologia, Chirurgia...","{""Da consigliare a parenti e amici"": 5.0, ""Pul...","{""Neuropsichiatria Infantile"": 5.0}"
2,Ospedale Filippo del Ponte di Varese - ASST Se...,"Via Filippo del Ponte, 19 - 21100\n Vares...",https://www.micuro.it/strutture/ospedale-filip...,5.0,2.0,14,"Cardiologia, Chirurgia Generale, Fisiopatologi...","{""Da consigliare a parenti e amici"": 5.0, ""Pul...","{""Ginecologia"": 5.0, ""Neonatologia"": 5.0, ""Neu..."
3,IRCCS Maugeri Lumezzane,"Via Giuseppe Mazzini, 129 - 25065\n Lumez...",https://www.micuro.it/strutture/istituto-scien...,5.0,1.0,27,"Allergologia, Analisi di Laboratorio, Angiolog...","{""Da consigliare a parenti e amici"": 5.0, ""Pul...",{}
4,Ospedale di Seregno - ASST Brianza,"Via Verdi, 2 - 20831\n Seregno (MB)",https://www.micuro.it/strutture/ospedale-di-se...,5.0,1.0,7,"Dietologia e Nutrizione Clinica, Endocrinologi...","{""Da consigliare a parenti e amici"": 5.0, ""Pul...",{}


In [28]:
outputName = OUTPUT_DIR / "micuro_lombardia_strutture.csv"
df.to_csv(outputName, index=False, encoding="utf-8-sig")